In [ ]:
!pip install -q google-cloud-storage pandas

In [ ]:
PROJECT_ID = "aiproject-476812"  # @param {type:"string"}
REGION = "us-east1"  # @param {type:"string"}


In [ ]:
import pandas as pd
import json
from google.cloud import storage

## Step 2: Configure your bucket and file paths
# EDIT THESE VALUES:
BUCKET_NAME = "aiproject_data_2946321"
INPUT_TSV_PATH = "menu_data_edited.tsv"
OUTPUT_JSONL_PATH = "menu_zh_en.jsonl"     # Final JSONL for Modele Training

## Step 3: Authenticate
from google.colab import auth
auth.authenticate_user()

## Step 4: Download TSV from Cloud Storage
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(INPUT_TSV_PATH)
blob.download_to_filename("menu_data_edited.tsv")
print(f"✓ Downloaded {INPUT_TSV_PATH}")

## Step 5: Load and validate
df = pd.read_csv("menu_data_edited.tsv", sep='\t', header=None,
                 names=['zh','en'], dtype=str, on_bad_lines='skip')
df = df[['zh','en']].dropna()
df['zh'] = df['zh'].str.strip()
df['en'] = df['en'].str.strip()
print(f"Total rows: {len(df)}")

## Step 6: Convert to JSONL (Gemini SFT format)
with open("menu_zh_en.jsonl", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        record = {
            "contents": [
                {"role": "user", "parts": [{"text": f"Translate to English:\n{row['zh']}"}]},
                {"role": "model", "parts": [{"text": row['en']}]}
            ]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"✓ JSONL created with {len(df)} examples")

## Step 7: Upload to Cloud Storage
bucket.blob(OUTPUT_JSONL_PATH).upload_from_filename("menu_zh_en.jsonl")
print(f"✓ Uploaded to gs://{BUCKET_NAME}/{OUTPUT_JSONL_PATH}")

print("\n🎉 Done! Use this path for Gemini supervised fine-tuning:")
print(f"   gs://{BUCKET_NAME}/{OUTPUT_JSONL_PATH}")

## Step 8: Preview first 3 records
print("\nFirst 3 JSONL records:")
with open("menu_zh_en.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(f"\nRecord {i+1}:")
        print(json.dumps(json.loads(line), indent=2, ensure_ascii=False))


New data prepration

In [ ]:
!pip install -q google-cloud-storage pandas scikit-learn

In [ ]:

import json
from google.cloud import storage
import random



BUCKET_NAME = "ai-project-backup"
BUCKETDIR = "datasets/"

with open("menu_zh_en_final_ready.jsonl", "r", encoding="utf-8") as f:
    all_data = [json.loads(line) for line in f]

random.seed(42)
random.shuffle(all_data)

# 3. Split the data 
train_data = all_data[:85]
val_data = all_data[85:]

# 4. Write them to separate files
with open("menu_zh_en_train.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

with open("menu_zh_en_validation.jsonl", "w", encoding="utf-8") as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Created Training Set: {len(train_data)} records")
print(f"Created Validation Set: {len(val_data)} records")

# 5. Upload both straight to your Cloud Bucket
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

bucket.blob(BUCKETDIR + "menu_zh_en_train.jsonl").upload_from_filename("menu_zh_en_train.jsonl")
bucket.blob(BUCKETDIR + "menu_zh_en_validation.jsonl").upload_from_filename("menu_zh_en_validation.jsonl")

print("Training and validation sets created and uploaded!")